In [1]:
import sys
import os
from dotenv import load_dotenv

# Sobe um nível no diretório para alcançar a pasta raiz BLUADIAGNOSTICS
caminho_raiz = os.path.abspath(os.path.join(os.getcwd(), '..'))
if caminho_raiz not in sys.path:
    sys.path.append(caminho_raiz)

# Carrega a chave OLLAMA_API_KEY do arquivo .env (Obrigatório para o servidor remoto)
load_dotenv()

print("✅ Ambiente configurado. Caminhos e chaves de API carregados com sucesso.")

✅ Ambiente configurado. Caminhos e chaves de API carregados com sucesso.


In [2]:
from src.guardrails.red_flags import detectar_red_flags
from src.guardrails.scope_validator import validar_escopo
from src.guardrails.moderation import moderar_conteudo

print("--- TESTE DE RED FLAGS ---")
print("Cenário Emergência:", detectar_red_flags("Doutor, estou com uma dor no peito muito forte e falta de ar.")) 
print("Cenário Normal:", detectar_red_flags("Minha garganta está arranhando há dois dias.")) 

print("\n--- TESTE DE ESCOPO ---")
print("Fora do escopo:", validar_escopo("Qual a receita de bolo de chocolate?")) 
print("Dentro do escopo:", validar_escopo("Quais os sintomas da gripe?")) 

print("\n--- TESTE DE MODERAÇÃO ---")
print("Ofensivo:", moderar_conteudo("Este aplicativo é uma bosta, palavrao1!")) 
print("Respeitoso:", moderar_conteudo("Poderia me ajudar com minha receita?"))

--- TESTE DE RED FLAGS ---
Cenário Emergência: True
Cenário Normal: False

--- TESTE DE ESCOPO ---
Fora do escopo: False
Dentro do escopo: True

--- TESTE DE MODERAÇÃO ---
Ofensivo: False
Respeitoso: True


In [3]:
from src.rag.vector_store import processar_conhecimento

print("Iniciando a indexação dos documentos médicos...")
processar_conhecimento()

Iniciando a indexação dos documentos médicos...
📂 [RAG] Processando base de conhecimento...
🔍 Buscando documentos em: /Users/christiandiaz/bluadiagnostics/data/knowledge_base


/Users/christiandiaz/bluadiagnostics/src/rag/embeddings.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  return OllamaEmbeddings(


[RAG] Base indexada com 20 fragmentos em /Users/christiandiaz/bluadiagnostics/data/chroma_db.


In [4]:
from src.rag.retriever import buscar_contexto_clinico

pergunta_paciente = "Quais as indicações e cuidados ao tomar amoxicilina?"
print(f"🔍 Buscando contexto para: '{pergunta_paciente}'\n")

# Chama a função que agora usa o embedding remoto via servidor do professor
resultado_rag = buscar_contexto_clinico(pergunta_paciente)

print("📚 FONTES RECUPERADAS (Para a Interface Visual):")
for fonte in resultado_rag["fontes_interface"]:
    print(f"- {fonte}")

print("\n📄 CONTEXTO EXTRAÍDO (Para o modelo gpt-oss:120b ler):")
# Imprimimos apenas os primeiros 500 caracteres para manter a tela limpa
print(resultado_rag["contexto_llm"][:500] + "\n\n[...texto continua...]")

🔍 Buscando contexto para: 'Quais as indicações e cuidados ao tomar amoxicilina?'

📚 FONTES RECUPERADAS (Para a Interface Visual):
- /Users/christiandiaz/bluadiagnostics/data/knowledge_base/cartilhas/cartilha_saude_mental.md
- /Users/christiandiaz/bluadiagnostics/data/knowledge_base/cartilhas/cartilha_saude_mental.md
- /Users/christiandiaz/bluadiagnostics/data/knowledge_base/cartilhas/cartilha_saude_mental.md
- /Users/christiandiaz/bluadiagnostics/data/knowledge_base/cartilhas/cartilha_saude_mental.md
- /Users/christiandiaz/bluadiagnostics/data/knowledge_base/cartilhas/cartilha_saude_mental.md
- /Users/christiandiaz/bluadiagnostics/data/knowledge_base/bulas/bula_amoxicilina.md

📄 CONTEXTO EXTRAÍDO (Para o modelo gpt-oss:120b ler):
[Extraído de: /Users/christiandiaz/bluadiagnostics/data/knowledge_base/cartilhas/cartilha_saude_mental.md]
# CARTILHA BLUA: SAÚDE MENTAL E CONTROLE DO ESTRESSE
O estresse crônico não afeta só o humor, mas pode causar palpitações, insônia e dores musculares.
Di

/Users/christiandiaz/bluadiagnostics/src/rag/retriever.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [5]:
from src.graph.builder import compilar_grafo
from langchain_core.messages import HumanMessage

# Compila todos os agentes (Supervisor, Triagem, Escalada) e RAG juntos
app = compilar_grafo()

def simular_chat(mensagem):
    print(f"\n" + "="*60)
    print(f"👤 PACIENTE: {mensagem}")
    print("="*60)
    
    estado_inicial = {"messages": [HumanMessage(content=mensagem)]}
    config = {"configurable": {"thread_id": "demo_notebook"}}
    
    # Inicia a jornada no LangGraph
    resultado = app.invoke(estado_inicial, config)
    
    if resultado.get("red_flag_detectada"):
        print("🚨 FLAG DE EMERGÊNCIA ACIONADA NO ESTADO DO GRAFO!")
        
    print(f"\n🤖 RESPOSTA FINAL DO SISTEMA:")
    print(resultado['messages'][-1].content)

# --- OS 3 CENÁRIOS DE TESTE ---

# 1. Teste de Guardrail (Deve ser barrado pelo Supervisor sem gastar tokens)
simular_chat("Quero saber qual time de futebol ganhou hoje.") 

# 2. Teste de Red Flag (Deve ir para Triagem e acionar o Agente de Escalada)
simular_chat("Doutor, estou com uma dor no peito muito forte e falta de ar!") 

# 3. Teste de Raciocínio Clínico Real (RAG Local + LLM 120B na Nuvem)
# Aguarde alguns segundos após rodar este, pois fará uma requisição real à API externa.
simular_chat("Quais as indicações e cuidados ao tomar amoxicilina?")

/Users/christiandiaz/bluadiagnostics/.venv/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



👤 PACIENTE: Quero saber qual time de futebol ganhou hoje.
👔 [SUPERVISOR] Analisando o estado da prancheta...
🚫 [SUPERVISOR] Conteúdo fora de escopo ou ofensivo. Encerrando.

🤖 RESPOSTA FINAL DO SISTEMA:
Desculpe, só posso ajudar com questões médicas respeitosas relacionadas à Care Plus.

👤 PACIENTE: Doutor, estou com uma dor no peito muito forte e falta de ar!
👔 [SUPERVISOR] Analisando o estado da prancheta...
👔 [SUPERVISOR] Contexto seguro. Direcionando para TRIAGEM.
🩺 [TRIAGEM] Assumindo o atendimento e invocando gpt-oss:120b...
⚠️ [TRIAGEM] Sintoma crítico detectado!
🚨 [ESCALADA] Ativando protocolo de emergência humano!
🚨 FLAG DE EMERGÊNCIA ACIONADA NO ESTADO DO GRAFO!

🤖 RESPOSTA FINAL DO SISTEMA:
⚠️ Por segurança, interrompemos esta triagem virtual. Por favor, dirija-se imediatamente ao Pronto-Socorro mais próximo ou ligue para o SAMU (192).

👤 PACIENTE: Quais as indicações e cuidados ao tomar amoxicilina?
👔 [SUPERVISOR] Analisando o estado da prancheta...
👔 [SUPERVISOR] Contexto

/Users/christiandiaz/bluadiagnostics/src/agents/triagem.py:13: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  return ChatOllama(



🤖 RESPOSTA FINAL DO SISTEMA:
**Amoxicilina – o que você precisa saber antes de iniciar o tratamento**

| **Indicações (para as quais a amoxicilina pode ser prescrita)** | **Cuidados essenciais** |
|---|---|
| • Infecções bacterianas do trato respiratório – como pneumonias e sinusites.<br>• Infecções do trato urinário.<br>• Infecções de pele e tecidos moles (por exemplo, celulite, impetigo). | • **Receita obrigatória** – a amoxicilina é um antibiótico de controle especial. Só pode ser usada mediante receita assinada por um médico da Care Plus e, quando necessário, a receita deve ser retida na farmácia.<br>• **Complete o ciclo** – mesmo que os sintomas melhorem antes do término, continue o tratamento até a última dose prescrita para evitar recaídas e resistência bacteriana.<br>• **Alergia a penicilinas** – informe ao médico se já teve reação alérgica (erupções, inchaço, dificuldade para respirar).<br>• **Reações adversas comuns** – náuseas, vômitos, diarreia ou erupções cutâneas leves. 